# <font color="steelblue">29. Contrastes de hipótesis: la lógica del p-valor</font>

*Unidad 5. Conceptos básicos de inferencia estadística* · Sesión teórica 29 de 30

**Autoría**:

*   Fernando Borrás Rocher (f.borras@umh.es), autor de la idea original
*   Juan Francisco Monge Ivars
*   José Santiago García Cremades

Departamento de Estadística, Matemáticas e Informática. Universidad Miguel Hernández de Elche.

Basado en [SIMUSTAT](https://unidigitalsimustat.umh.es/) (proyecto UNIDIGITAL-UMH, CC BY-SA 4.0).

**Asignatura**: Estadística (1200), 1.º curso, Grado en Periodismo.

**Licencia**: <a rel="license" href="http://creativecommons.org/licenses/by-sa/4.0/"><img alt="CC BY-SA 4.0" style="border-width:0" src="https://i.creativecommons.org/l/by-sa/4.0/88x31.png" /></a> Creative Commons Atribución-CompartirIgual 4.0.

> ⚠️ **Los datos de este curso son simulados.** Imitan la forma de fuentes reales (INE, CIS, audímetros, agencias de verificación), pero ninguna cifra es real. Úsalos para aprender; para publicar, ve siempre a la fuente oficial.

## <font color='steelblue'>Antes de empezar: configura el cuaderno</font>

Ejecuta **una vez** la celda de abajo (botón ▶). Instala la librería de las preguntas, descarga los datos (`encuesta_medios.csv`) y las preguntas de autoevaluación. Si todo va bien verás **`Cuaderno configurado`**.

> 💡 Si Colab se desconecta, los ficheros se borran: vuelve a ejecutar esta celda.

In [ ]:
#@title <font color="steelblue" size="+1"> Ejecutar configuración { display-mode: "form" }
import os, sys, json, base64, subprocess, urllib.request

RAW = "https://raw.githubusercontent.com/fborrasumh/statperiodismo/main/"
DATOS = ["encuesta_medios.csv"]
PREGUNTAS = ["q_P29_1.json", "q_P29_2.json", "q_P29_3.json", "q_P29_4.json"]

try:
    from jupyterquiz import display_quiz
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "jupyterquiz==2.9.6.4"], check=True)
    from jupyterquiz import display_quiz

for carpeta, ficheros in (("datos/", DATOS), ("autoeval/", PREGUNTAS)):
    for f in ficheros:
        if not os.path.exists(f):
            urllib.request.urlretrieve(RAW + carpeta + f, f)

def pregunta(fichero):
    """Muestra una pregunta interactiva de autoevaluación."""
    display_quiz(json.loads(base64.b64decode(open(fichero, "rb").read()).decode("utf8")))

print("Cuaderno configurado")

## <font color='steelblue'>Introducción</font>

«Un estudio demuestra que...»: detrás de muchas de estas noticias hay un **contraste de hipótesis** y un **p-valor**. Es la herramienta que usa la ciencia para decidir si un resultado puede deberse al azar. También es una de las más malinterpretadas. Entender su lógica te permitirá leer estudios con criterio y no repetir sus exageraciones.

**Qué necesitas saber antes**: los cuadernos 26 y 27.

## <font color='steelblue'>Qué vas a aprender</font>

Al terminar este cuaderno sabrás:

1. Plantear una hipótesis nula y una alternativa.
2. Calcular e interpretar un p-valor, por simulación y con fórmula.
3. Distinguir los errores de tipo I y de tipo II.
4. Reconocer usos engañosos: «significativo» no es «importante» y el p-hacking.

## <font color='steelblue'>Contenidos</font>

1. La lógica: un juicio a la hipótesis nula
2. El p-valor por simulación
3. Dos tipos de error
4. Significativo no es importante

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

plt.rcParams['figure.figsize'] = (10, 4)
pd.set_option('display.max_columns', 20)

## <font color='steelblue'>1. La lógica: un juicio a la hipótesis nula</font>

Un contraste funciona como un juicio:

| Juicio | Contraste |
| :-- | :-- |
| Presunción de inocencia | **Hipótesis nula** $H_0$: no hay efecto, no hay diferencia, todo es azar |
| La acusación | **Hipótesis alternativa** $H_1$: sí hay efecto |
| Las pruebas | Los datos de la muestra |
| «Más allá de toda duda razonable» | El **p-valor** es muy pequeño |
| Absuelto no significa inocente | No rechazar $H_0$ no demuestra que $H_0$ sea cierta |

Ejemplo: una cabecera afirma que **más de la mitad** de sus lectores la prefiere en digital. En una muestra de 400 lectores, 222 (55,5 %) lo dicen. ¿Basta para afirmarlo, o podría ser azar con un 50 % real?

- $H_0$: $p = 0{,}5$ &nbsp;&nbsp; $H_1$: $p > 0{,}5$

### <font color='steelblue'>Autoevaluación</font>

Ejecuta la celda y escoge la respuesta correcta (**solo hay una**):

In [ ]:
#@title <font color='steelblue'> Pregunta interactiva { display-mode: "form" }
pregunta("q_P29_1.json")

## <font color='steelblue'>2. El p-valor por simulación</font>

El **p-valor** es la probabilidad de obtener un resultado **igual o más extremo** que el observado **si $H_0$ fuera cierta**. Lo calculamos simulando muchas muestras de 400 lectores con un 50 % real:

In [ ]:
rng = np.random.default_rng(29)
n, observados = 400, 222
sim = rng.binomial(n, 0.5, size=100_000)
p_valor_sim = (sim >= observados).mean()

plt.hist(sim, bins=50, color="lightgrey")
plt.axvline(observados, color="firebrick", label=f"Observado: {observados}")
plt.xlabel("Lectores que prefieren digital en muestras de 400 (si p = 0,5)")
plt.legend()
plt.title(f"p-valor simulado = {p_valor_sim:.4f}")
plt.show()

Con la fórmula exacta (`binomtest`) obtenemos casi lo mismo:

In [ ]:
res = stats.binomtest(observados, n, p=0.5, alternative="greater")
print(f"p-valor exacto: {res.pvalue:.4f}")

**Cómo leerlo**: si de verdad la preferencia fuera del 50 %, obtener 222 o más de 400 pasaría solo en torno a un 1,5 % de las muestras. Es poco probable, así que **rechazamos $H_0$** al nivel habitual del 5 %: los datos apoyan que más de la mitad prefiere el digital.

El umbral (normalmente 0,05) se llama **nivel de significación** $\alpha$. Si p < α, se dice que el resultado es **estadísticamente significativo**.

### <font color='steelblue'>Autoevaluación</font>

Ejecuta la celda y escoge la respuesta correcta (**solo hay una**):

In [ ]:
#@title <font color='steelblue'> Pregunta interactiva { display-mode: "form" }
pregunta("q_P29_2.json")

## <font color='steelblue'>3. Dos tipos de error</font>

| | $H_0$ es cierta | $H_0$ es falsa |
| :-- | :-- | :-- |
| **Rechazo $H_0$** | Error de **tipo I** (falso positivo), probabilidad α | Acierto |
| **No rechazo $H_0$** | Acierto | Error de **tipo II** (falso negativo) |

Con α = 0,05, **1 de cada 20** contrastes en los que no hay ningún efecto saldrá «significativo» por puro azar.

**Qué vamos a hacer**: simular 100 estudios en los que **no hay ningún efecto real** y ver cuántos salen significativos. Esperamos unos 5.

In [ ]:
significativos = 0
for estudio in range(100):
    grupo_a = rng.normal(50, 10, 100)
    grupo_b = rng.normal(50, 10, 100)          # misma media: no hay efecto
    p = stats.ttest_ind(grupo_a, grupo_b).pvalue
    significativos += p < 0.05
print(f"Estudios 'significativos' sin ningún efecto real: {significativos} de 100")

### <font color='steelblue'>Autoevaluación</font>

Ejecuta la celda y escoge la respuesta correcta (**solo hay una**):

In [ ]:
#@title <font color='steelblue'> Pregunta interactiva { display-mode: "form" }
pregunta("q_P29_3.json")

## <font color='steelblue'>4. Significativo no es importante</font>

Dos trampas que verás en notas de prensa de estudios:

1. **Con muestras enormes, cualquier diferencia minúscula es significativa.** Un p-valor pequeño dice que el efecto probablemente **existe**, no que sea **grande** ni **importante**. Pregunta siempre por el **tamaño del efecto** y su intervalo de confianza.
2. **P-hacking**: si se prueban 20 cruces de variables, alguno saldrá significativo por azar (lo acabas de ver). Desconfía de los estudios que «descubren» un efecto sin haberlo planteado de antemano.

In [ ]:
for n_ in [100, 10_000, 1_000_000]:
    a = rng.normal(50.0, 10, n_)
    b = rng.normal(50.3, 10, n_)                # diferencia real minúscula: 0,3 puntos
    print(f"n = {n_:>9,} por grupo: diferencia = {b.mean() - a.mean():.2f}, p-valor = {stats.ttest_ind(a, b).pvalue:.4f}")

> **Para la noticia**: «el estudio encuentra una diferencia estadísticamente significativa, aunque pequeña: de 0,3 puntos sobre 100». Mejor eso que «un estudio demuestra que...».

### <font color='steelblue'>Autoevaluación</font>

Ejecuta la celda y marca **todas** las respuestas correctas:

In [ ]:
#@title <font color='steelblue'> Pregunta interactiva { display-mode: "form" }
pregunta("q_P29_4.json")

## <font color='steelblue'>Reto aprendiz</font>

Resuelve estos ejercicios en la celda vacía de debajo de cada enunciado. Si te atascas, vuelve a la sección indicada. La solución está plegada: **inténtalo antes de mirarla**.

### **Ejercicio 1** *(sección 2)*

En la encuesta (n = 1.200), ¿hay evidencia de que **más del 25 %** ha creído alguna vez un bulo? Plantea las hipótesis y calcula el p-valor con `binomtest`.

<details><summary>Ver solución</summary>

```python
enc = pd.read_csv("encuesta_medios.csv")
k = (enc["ha_creido_bulo"] == "Sí").sum()
stats.binomtest(k, len(enc), p=0.25, alternative="greater").pvalue
```
</details>

### **Ejercicio 2** *(sección 2)*

Repite el ejemplo de los lectores con una muestra de 100 y 55 respuestas a favor (también el 55 %). ¿Es significativo? ¿Por qué cambia?

<details><summary>Ver solución</summary>

```python
stats.binomtest(55, 100, p=0.5, alternative="greater").pvalue
```

No: con 100 casos, un 55 % es compatible con el azar. El mismo porcentaje con más datos da más evidencia.
</details>

### **Ejercicio 3** *(sección 3)*

Repite la simulación de los estudios sin efecto 1.000 veces en lugar de 20. ¿Qué proporción sale significativa?

<details><summary>Ver solución</summary>

```python
sig = 0
for _ in range(1000):
    sig += stats.ttest_ind(rng.normal(50, 10, 100), rng.normal(50, 10, 100)).pvalue < 0.05
sig / 1000
```
</details>

### **Ejercicio 4** *(sección 4)*

Busca una noticia reciente que diga «un estudio demuestra...». ¿Indica el tamaño del efecto, el tamaño de la muestra y si fue un experimento?

<details><summary>Ver idea clave</summary>

Ejercicio abierto para comentar en clase.
</details>

## <font color='steelblue'>Resumen: ideas clave</font>

- Un **contraste** plantea $H_0$ (no hay efecto) frente a $H_1$ y usa los datos para decidir.
- El **p-valor** es la probabilidad de un resultado tan extremo o más **si $H_0$ fuera cierta**; si es menor que α (0,05), se rechaza $H_0$.
- **Error de tipo I**: falso positivo (probabilidad α). **Error de tipo II**: falso negativo.
- «Significativo» **no** es «importante»: pide el tamaño del efecto y su intervalo.
- Muchos contrastes producen falsos positivos (**p-hacking**).

## <font color='steelblue'>Glosario</font>

| Término | Significado |
| :-- | :-- |
| **Hipótesis nula** | Afirmación de que no hay efecto, que se pone a prueba. |
| **p-valor** | Probabilidad de un resultado igual o más extremo si H0 es cierta. |
| **Nivel de significación** | Umbral α por debajo del cual se rechaza H0. |
| **Error de tipo I** | Rechazar H0 siendo cierta. |
| **Error de tipo II** | No rechazar H0 siendo falsa. |
| **P-hacking** | Probar muchos análisis hasta encontrar uno significativo. |

## <font color='steelblue'>Errores frecuentes y cómo resolverlos</font>

| Síntoma | Causa probable | Solución |
| :-- | :-- | :-- |
| `AttributeError: binom_test` | Esa función se eliminó de SciPy | Usa `stats.binomtest(...).pvalue` |
| `NameError: name 'pd' is not defined` | No se ejecutó la celda de importación | Ejecuta las celdas desde el principio, en orden |
| `FileNotFoundError` al leer un fichero | No se ejecutó la configuración o Colab se reinició | Ejecuta la celda *Configura el cuaderno* |
| La pregunta de autoevaluación no aparece | Falta la configuración | Ejecuta la celda de configuración y luego la pregunta |

## <font color='steelblue'>Para seguir</font>

En el cuaderno 30, el último, compararás dos grupos: proporciones, tablas de contingencia y medias.

**Siguiente cuaderno**: [30. Comparar dos grupos: proporciones, tablas y medias](https://colab.research.google.com/github/fborrasumh/statperiodismo/blob/main/cuadernos/P30_comparar_dos_grupos.ipynb)